# Half-system centered-covariance spectrum: $20\times32$ hard wall

All 100 pure-state endpoints at cycle 64, $\alpha_1=1$, $\alpha_2=30$, and overcomplete Wannier (OW) range $n_{\mathrm{shell}}=1$. The fixed spatial subsystem contains all $x$, $y=0,\ldots,15$, and both orbitals. Inputs are read-only; all outputs stay beside this notebook.

## CPU allocation
Edit the inclusive CPU range below before running. The default uses four available CPUs; allocation and BLAS threads are bounded before numerical imports.

In [ ]:
import os
# Preserve the original kernel affinity so this cell can be rerun with a new range.
if "AVAILABLE_CPUS" not in globals():
    AVAILABLE_CPUS = sorted(os.sched_getaffinity(0))
CPU_RANGE = (AVAILABLE_CPUS[0], AVAILABLE_CPUS[min(3, len(AVAILABLE_CPUS)-1)])
selected_cpus = list(range(CPU_RANGE[0], CPU_RANGE[1] + 1))
if not selected_cpus or not set(selected_cpus).issubset(AVAILABLE_CPUS):
    raise ValueError("CPU_RANGE must be an inclusive range of available CPUs")
os.sched_setaffinity(0, selected_cpus)
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = str(len(selected_cpus))
from threadpoolctl import threadpool_limits
blas_limit = threadpool_limits(limits=len(selected_cpus))
print("Allocated CPUs:", selected_cpus)

## Estimator and convention
Here $C_A=F_AF_A^\dagger$ denotes the occupation matrix and $G_A=2C_A-I_{640}$ the **centered** covariance, matching the saved-engine convention (the paper uses $G$ for the uncentered correlation matrix).

For each independent Born trajectory, restrict the **active** occupied-frame columns to the subsystem and diagonalize $G_A$. Keep all 640 eigenvalues, including the endpoints $\pm1$. Pool the 100 spectra only after diagonalization; no ensemble-averaged matrix is diagonalized and no translated cuts are averaged.

For bin width $\Delta\lambda$, the density is $\rho_j=n_j/(64000\Delta\lambda)$, so $\sum_j\rho_j\Delta\lambda=1$. Numerical excursions beyond $[-1,1]$ smaller than $10^{-8}$ are clipped **only for histogramming**; cached spectra remain raw.

## Configuration and input provenance
The selected campaign identity is frozen below. Four completion receipts and their NPZ contents must agree before analysis.

In [ ]:
from pathlib import Path
import hashlib, json, time, platform
from datetime import datetime, timezone
import numpy as np
import scipy
from scipy.linalg import eigvalsh
from tqdm.auto import tqdm
from IPython.display import display, Markdown

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "PROJECT_ADMIN/REPO_POLICY.md").is_file())
BUNDLE = REPO_ROOT / "00_WORKSPACE/CURRENT/final_production_new_designs/09_pure_tangent_replay_acquisition"
REVISION = "pure_tangent_replay_nx20_ny24-32_hard-soft_a1-1-3_s100_v1"
DATA_DIR = BUNDLE / "gpu_data" / REVISION / "hard/Ny032/alpha1_1"
OUTPUT_DIR = BUNDLE / "analysis_outputs/half_system_centered_spectrum_n20x32_hard_alpha1_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_CONFIG_HASH = "d1a7c212d8f2c06b0f774af14272a2c71b9d0482240f0c4c43be75a9cd494be4"
NX, NY, AY, SAMPLE_COUNT = 20, 32, 16, 100
BOUND_TOL, HERMITICITY_TOL, TRACE_TOL = 1e-8, 1e-12, 1e-9
FORCE_RECOMPUTE = False
CACHE = OUTPUT_DIR / "centered_spectra.npz"
DIAGNOSTICS = OUTPUT_DIR / "diagnostics.json"
indices = np.array([2*NX*y + 2*x + orbital
                    for y in range(AY) for x in range(NX) for orbital in range(2)])
MODES = len(indices)
assert MODES == 640 and np.array_equal(indices, np.arange(640))

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha256(path):
    with Path(path).open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

receipts = sorted(DATA_DIR.glob("*.complete.json"))
require(len(receipts) == 4, "Expected exactly four completion receipts")
require(len(list(DATA_DIR.glob("*.npz"))) == 4, "Expected exactly four result batches")
inputs, ids, source_hashes = [], [], None
for receipt in tqdm(receipts, desc="Verify input checksums", unit="batch"):
    r = json.loads(receipt.read_text())
    for key, value in {"Nx": NX, "Ny": NY, "cycles": 64, "construction": "hard",
                       "alpha_1": 1.0, "alpha_2": 30.0, "nshell": 1,
                       "configuration_sha256": EXPECTED_CONFIG_HASH,
                       "sampling_revision": REVISION, "status": "complete",
                       "canonical_entry_point": "classA_U1FGTN_gpu.run_markov_circuit"}.items():
        require(r.get(key) == value, f"Receipt mismatch: {receipt.name}: {key}")
    filename = r["result_filename"]
    require(Path(filename).name == filename, "Result filename must be local to data directory")
    result = DATA_DIR / filename
    require(receipt.name == result.stem + ".complete.json", "Receipt filename mismatch")
    digest = sha256(result)
    require(result.stat().st_size == r["result_bytes"], "Result byte count mismatch")
    require(digest == r["result_sha256"], "Result SHA-256 mismatch")
    require(bool(r["source_hashes"]), "Missing source identities")
    if source_hashes is None:
        source_hashes = r["source_hashes"]
    require(r["source_hashes"] == source_hashes, "Inconsistent executed source identities")
    with np.load(result, allow_pickle=False) as z:
        for key, value in {"Nx": NX, "Ny": NY, "cycles_total": 64,
                           "construction": "hard", "alpha_1": 1.0, "alpha_2": 30.0,
                           "nshell": 1, "dtype": "complex128", "sequence": "raster_y",
                           "configuration_sha256": EXPECTED_CONFIG_HASH,
                           "sampling_revision": REVISION,
                           "canonical_entry_point": r["canonical_entry_point"],
                           "task_id": r["task_id"], "batch_seed": r["seed"]}.items():
            require(z[key].item() == value, f"NPZ metadata mismatch: {key}")
        sample_ids = z["case_sample_indices"].tolist()
        require(sample_ids == r["case_sample_indices"] and len(sample_ids) == 25,
                "Sample identity mismatch")
        require(z["global_sample_indices"].tolist() == r["global_sample_indices"],
                "Global sample identity mismatch")
        require(json.loads(z["source_hashes_json"].item()) == source_hashes,
                "NPZ source identity mismatch")
    ids.extend(sample_ids)
    inputs.append({"result": str(result.relative_to(REPO_ROOT)), "sha256": digest,
                   "bytes": r["result_bytes"], "receipt_sha256": sha256(receipt),
                   "receipt": str(receipt.relative_to(REPO_ROOT)),
                   "sample_ids": sample_ids, "seed": r["seed"], "task_id": r["task_id"]})
require(sorted(ids) == list(range(SAMPLE_COUNT)), "Missing or duplicate sample IDs")
contract = {"schema": "half_system_centered_spectrum_v1", "inputs": inputs,
            "configuration_sha256": EXPECTED_CONFIG_HASH, "source_hashes": source_hashes,
            "subsystem_indices": indices.tolist(), "formula": "G_A=2*F_A@F_A.conj().T-I",
            "bounds_tolerance": BOUND_TOL, "hermiticity_tolerance": HERMITICITY_TOL,
            "trace_absolute_tolerance": TRACE_TOL}
identity = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()
display(Markdown(f"**Verified:** {SAMPLE_COUNT} independent pure-state trajectories; "
                 f"hard wall; {NX}×{NY}; α₁=1, α₂=30; nshell=1; cycle 64; "
                 f"raster-y; perfect correction; no postselection. "
                 f"**Subsystem:** all x, y=0…15, both orbitals ({MODES} modes)."))
print("Input:", DATA_DIR)
print("Output:", OUTPUT_DIR)
print("Configuration SHA-256:", EXPECTED_CONFIG_HASH)

## Sample-resolved spectra
Read one batch at a time. Check frame finiteness and orthonormality, raw matrix Hermiticity, spectral bounds, and the trace identity. Reuse the compact cache only when input identity and cache checksum match.

In [ ]:
start = time.perf_counter()
reuse = False
if CACHE.exists() and DIAGNOSTICS.exists() and not FORCE_RECOMPUTE:
    previous = json.loads(DIAGNOSTICS.read_text())
    reuse = (previous.get("analysis_identity") == identity
             and previous.get("spectra_sha256") == sha256(CACHE))
if reuse:
    with np.load(CACHE, allow_pickle=False) as z:
        require(z["analysis_identity"].item() == identity, "Cached identity mismatch")
        require(np.array_equal(z["sample_ids"], np.arange(SAMPLE_COUNT)), "Cached sample mismatch")
        spectra = z["eigenvalues"]
    diagnostics = previous
    print("Loaded verified spectra cache; no eigendecompositions repeated.")
else:
    spectra = np.empty((SAMPLE_COUNT, MODES), dtype=np.float64)
    sample_checks = []
    crosscheck = None
    progress = tqdm(total=SAMPLE_COUNT, desc="Half-system eigenspectra", unit="trajectory")
    try:
        for info in inputs:
            with np.load(REPO_ROOT / info["result"], allow_pickle=False) as z:
                frames, ranks = z["final_frame"], z["final_ranks"]
            require(frames.dtype == np.complex128 and frames.shape[:2] == (25, 1280),
                    "Unexpected final-frame shape or dtype")
            require(ranks.shape == (25,) and np.issubdtype(ranks.dtype, np.integer),
                    "Invalid ranks array")
            for row, sample_id in enumerate(info["sample_ids"]):
                rank = int(ranks[row])
                require(0 <= rank <= frames.shape[2], "Invalid active-column rank")
                F = frames[row, :, :rank]
                require(np.isfinite(F).all(), f"Nonfinite frame: {sample_id}")
                gram_residual = float(np.max(np.abs(F.conj().T @ F - np.eye(rank)))) if rank else 0.0
                require(gram_residual <= BOUND_TOL, f"Nonorthonormal frame: {sample_id}")
                FA = F[indices, :]
                C = FA @ FA.conj().T
                G = 2*C - np.eye(MODES)
                require(np.isfinite(G).all(), f"Nonfinite covariance: {sample_id}")
                herm = float(np.max(np.abs(G-G.conj().T)))
                require(herm <= HERMITICITY_TOL, f"Non-Hermitian covariance: {sample_id}")
                values = eigvalsh((G+G.conj().T)/2, check_finite=True, driver="evr")
                require(values.shape == (MODES,) and np.isfinite(values).all(), "Invalid spectrum")
                require(values.min() >= -1-BOUND_TOL and values.max() <= 1+BOUND_TOL,
                        f"Spectrum outside numerical tolerance: {sample_id}")
                trace_error = float(abs(values.sum()-np.trace(G)))
                require(trace_error <= TRACE_TOL, f"Trace closure failed: {sample_id}")
                if sample_id == 0:
                    alternate = 2*eigvalsh(C, driver="evd")-1
                    crosscheck = float(np.max(np.abs(values-alternate)))
                    require(crosscheck <= 1e-10, "Occupation-spectrum cross-check failed")
                spectra[sample_id] = values
                sample_checks.append({"sample_id": sample_id, "rank": rank,
                                      "gram_residual": gram_residual, "hermiticity_residual": herm,
                                      "trace_error": trace_error, "raw_min": float(values.min()),
                                      "raw_max": float(values.max())})
                progress.update()
            del frames, ranks, F, FA, C, G
    finally:
        progress.close()
    temporary = CACHE.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, eigenvalues=spectra, sample_ids=np.arange(SAMPLE_COUNT),
                        subsystem_indices=indices, analysis_identity=np.array(identity))
    temporary.replace(CACHE)
    diagnostics = {"analysis_identity": identity, "contract": contract,
                   "created_utc": datetime.now(timezone.utc).isoformat(),
                   "spectra_sha256": sha256(CACHE), "sample_checks": sample_checks,
                   "sample0_crosscheck_max_error": crosscheck,
                   "calculation_seconds": time.perf_counter()-start,
                   "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                               "scipy": scipy.__version__, "cpus": selected_cpus},
                   "independent_sampling_unit": "whole Born trajectory",
                   "canonical_dynamics_entry_point": "classA_U1FGTN_gpu.run_markov_circuit",
                   "dynamics_rerun": False}
    DIAGNOSTICS.write_text(json.dumps(diagnostics, indent=2, allow_nan=False)+"\n")
require(spectra.shape == (100, 640) and np.isfinite(spectra).all(), "Invalid cached spectra")
print("Spectrum array:", spectra.shape, "raw extrema:", spectra.min(), spectra.max())

## Histogram and figure
Change `BIN_COUNT` in the next cell and rerun it plus the diagnostics cell to rebin cached spectra. The full spectral mass, including the endpoint peaks, is retained. Bar heights are probability density, not probability per bin.

In [ ]:
BIN_COUNT = 100
# Use the repository-proven OT1 compatibility shim for this TeX installation.
os.environ["TEXINPUTS"] = str(OUTPUT_DIR / "latex_support") + os.pathsep + os.environ.get("TEXINPUTS", "")
import matplotlib as mpl
import matplotlib.pyplot as plt

require(isinstance(BIN_COUNT, int) and BIN_COUNT > 0, "BIN_COUNT must be a positive integer")
raw = spectra.ravel()
require(raw.min() >= -1-BOUND_TOL and raw.max() <= 1+BOUND_TOL, "Spectrum outside bounds")
clipped = np.clip(raw, -1, 1)  # Do not modify the raw cache.
edges = np.linspace(-1, 1, BIN_COUNT+1)
counts, edges = np.histogram(clipped, bins=edges)
widths = np.diff(edges)
density = counts / (raw.size * widths)
area = float(np.sum(density*widths))
require(int(counts.sum()) == 64000, "Histogram lost eigenvalues")
require(abs(area-1) <= 1e-14, "Histogram density is not normalized")
np.savetxt(OUTPUT_DIR / "histogram.csv",
           np.column_stack([edges[:-1], edges[1:], counts, density]), delimiter=",",
           header="left_edge,right_edge,count,probability_density", comments="",
           fmt=["%.17g", "%.17g", "%d", "%.17g"])

mpl.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["CMU Sans Serif"],
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                     "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "text.usetex": True, "pdf.fonttype": 42,
                     "xtick.direction": "in", "ytick.direction": "in"})
fig, ax = plt.subplots(figsize=(3.375, 2.65))
ax.stairs(density, edges, fill=True, color="#3465a4", alpha=0.75, linewidth=0.7)
ax.set(xlim=(-1, 1), ylim=(0, None), xlabel=r"Centered eigenvalue $\lambda$",
       ylabel=r"Probability density $\rho(\lambda)$",
       title=r"$20\times32$ hard wall, $\alpha_1=1$, cycle 64")
ax.tick_params(top=True, right=True)
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.text(0.5, 0.95, r"$S=100$, $640$ modes/sample"+"\n"+f"{BIN_COUNT} bins; unit area",
        transform=ax.transAxes, ha="center", va="top", fontsize=8)
fig.tight_layout(pad=0.5)
fig.savefig(OUTPUT_DIR / "centered_spectrum_histogram.pdf")
plt.close(fig)
# Rasterize the exact LaTeX/vector PDF at 300 dpi; dvipng is not installed here.
import subprocess
from IPython.display import Image
subprocess.run(["pdftoppm", "-r", "300", "-singlefile", "-png",
                str(OUTPUT_DIR / "centered_spectrum_histogram.pdf"),
                str(OUTPUT_DIR / "centered_spectrum_histogram")], check=True)
display(Image(filename=str(OUTPUT_DIR / "centered_spectrum_histogram.png"), width=500))

caption = ("Half-system centered-covariance spectral density for 100 independent Born trajectories "
           "of the 20×32 hard-wall circuit (alpha_1=1, alpha_2=30, nshell=1), "
           "from pure half-filled random initialization to cycle 64; raster-y ordering, "
           "perfect correction, no postselection. The fixed subsystem is all x and y=0…15 "
           "with both orbitals. Each 640-mode covariance is diagonalized separately before "
           f"pooling all 64,000 eigenvalues into {BIN_COUNT} equal-width bins on [-1,1]. "
           "Density integrates to one; endpoint spectral mass is included. No uncertainty "
           "intervals or fits are shown. Raw spectra are preserved; only numerical "
           "out-of-range excursions within 1e-8 are clipped for binning.")
(OUTPUT_DIR / "figure_caption.txt").write_text(caption+"\n")
display(Markdown(caption))

## Mixed-mode spectrum: exclude numerical endpoints

Retain $|\lambda|<1-\epsilon$ with editable $\epsilon=10^{-8}$, to exclude pure modes and endpoint roundoff. Modes closer to the endpoints than this tolerance are numerically unresolved and excluded, not claimed to be exactly pure.

Normalize the retained pooled modes to unit area. This is a conditional density, so its normalization differs from the full-spectrum plot above; each retained mode has equal weight. A logarithmic density axis reveals the interior while retaining near-endpoint mixed modes. The bin-integrated density, not the area in logarithmic display coordinates, equals one.

In [ ]:
MIXED_TOL = 1e-8
MIXED_BIN_COUNT = 100
require(0 < MIXED_TOL < 1, "MIXED_TOL must lie in (0,1)")
require(isinstance(MIXED_BIN_COUNT, int) and MIXED_BIN_COUNT > 0, "Invalid bin count")
mixed_mask = np.abs(spectra) < 1-MIXED_TOL
mixed_values = spectra[mixed_mask]
require(mixed_values.size > 0, "No modes retained at this tolerance")
mixed_edges = np.linspace(-1, 1, MIXED_BIN_COUNT+1)
mixed_counts, mixed_edges = np.histogram(mixed_values, bins=mixed_edges)
mixed_widths = np.diff(mixed_edges)
mixed_density = mixed_counts/(mixed_values.size*mixed_widths)
mixed_area = float(np.sum(mixed_density*mixed_widths))
require(int(mixed_counts.sum()) == mixed_values.size, "Mixed-mode histogram lost values")
require(abs(mixed_area-1) <= 1e-14, "Mixed-mode density is not normalized")
np.savetxt(OUTPUT_DIR/"mixed_modes_histogram.csv",
           np.column_stack([mixed_edges[:-1], mixed_edges[1:], mixed_counts, mixed_density]),
           delimiter=",", header="left_edge,right_edge,count,conditional_probability_density",
           comments="", fmt=["%.17g", "%.17g", "%d", "%.17g"])
mixed_summary = {
    "endpoint_exclusion_tolerance": MIXED_TOL,
    "selection": "abs(lambda) < 1 - tolerance",
    "retained_modes": int(mixed_values.size),
    "excluded_modes": int(spectra.size-mixed_values.size),
    "retained_fraction": float(mixed_values.size/spectra.size),
    "retained_per_sample": mixed_mask.sum(axis=1).tolist(),
    "mean_retained_per_sample": float(mixed_mask.sum(axis=1).mean()),
    "bin_count": MIXED_BIN_COUNT, "density_integral": mixed_area,
    "normalization": "pooled retained modes; unit bin-integrated area",
    "axis_scale": "logarithmic probability density",
    "raw_minimum_retained": float(mixed_values.min()),
    "raw_maximum_retained": float(mixed_values.max()),
    "source_spectra_sha256": sha256(CACHE),
}
(OUTPUT_DIR/"mixed_modes_diagnostics.json").write_text(
    json.dumps(mixed_summary, indent=2, allow_nan=False)+"\n")

fig, ax = plt.subplots(figsize=(3.375, 2.65))
# NaN gaps for empty bins avoid inventing a positive density on a log axis.
positive_density = np.where(mixed_counts > 0, mixed_density, np.nan)
ax.stairs(positive_density, mixed_edges, color="#3465a4", linewidth=1)
ax.set_yscale("log")
ax.set(xlim=(-1, 1), xlabel=r"Centered eigenvalue $\lambda$",
       ylabel=r"Conditional density $\rho_{\mathrm{mixed}}(\lambda)$",
       title=r"Mixed modes: $20\times32$ hard wall")
ax.tick_params(top=True, right=True)
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
tol_tex = f"{MIXED_TOL:.0e}"
ax.text(0.5, 0.96, f"{mixed_values.size:,} retained modes; $S=100$"
        +"\n"+f"Endpoint tolerance {tol_tex}; unit area",
        transform=ax.transAxes, ha="center", va="top", fontsize=8)
fig.tight_layout(pad=0.5)
fig.savefig(OUTPUT_DIR/"mixed_modes_histogram.pdf")
plt.close(fig)
subprocess.run(["pdftoppm", "-r", "300", "-singlefile", "-png",
                str(OUTPUT_DIR/"mixed_modes_histogram.pdf"),
                str(OUTPUT_DIR/"mixed_modes_histogram")], check=True)
display(Image(filename=str(OUTPUT_DIR/"mixed_modes_histogram.png"), width=500))
mixed_caption = (
    "Conditional mixed-mode density for the same 100 independent Born trajectories, "
    "20x32 hard-wall alpha_1=1, alpha_2=30, nshell=1, pure half-filled initialization, "
    "perfect correction, no postselection, raster-y, cycle 64. "
    "Subsystem: all x, y=0..15, both orbitals; diagonalize each sample before pooling. "
    f"Retain abs(lambda)<1-{MIXED_TOL:g}: {mixed_values.size} of {spectra.size} modes "
    f"({mixed_values.size/spectra.size:.3%}). Normalize the {MIXED_BIN_COUNT}-bin "
    "histogram by retained count and bin width. Bin-integrated area is one. "
    "The density axis is logarithmic; empty bins are gaps, not pseudocounts. "
    "Excluded modes are unresolved at the chosen endpoint tolerance. "
    "No uncertainty intervals or fits are shown.")
(OUTPUT_DIR/"mixed_modes_caption.txt").write_text(mixed_caption+"\n")
display(Markdown(mixed_caption))

## Raw diagnostics and completion
The histogram normalization and numerical residuals below refer to all 100 samples; the alternate occupation-matrix eigensolver check uses sample 0.

In [ ]:
summary = {"samples": 100, "modes_per_sample": 640, "eigenvalues": int(raw.size),
           "verified_input_pairs": len(inputs), "bin_count": BIN_COUNT,
           "histogram_count": int(counts.sum()), "density_integral": area,
           "raw_minimum": float(raw.min()), "raw_maximum": float(raw.max()),
           "clipped_below_minus_one": int(np.count_nonzero(raw < -1)),
           "clipped_above_plus_one": int(np.count_nonzero(raw > 1)),
           "maximum_clipping_excursion": float(np.max(np.abs(raw-clipped))),
           "max_hermiticity_residual": max(x["hermiticity_residual"] for x in diagnostics["sample_checks"]),
           "max_gram_residual": max(x["gram_residual"] for x in diagnostics["sample_checks"]),
           "max_trace_error": max(x["trace_error"] for x in diagnostics["sample_checks"]),
           "sample0_crosscheck_max_error": diagnostics["sample0_crosscheck_max_error"],
           "spectra_sha256": sha256(CACHE), "all_checks_passed": True}
diagnostics["summary"] = summary
diagnostics["outputs"] = {name: sha256(OUTPUT_DIR/name) for name in
                          ["centered_spectra.npz", "histogram.csv", "centered_spectrum_histogram.pdf",
                           "centered_spectrum_histogram.png", "figure_caption.txt"]}
DIAGNOSTICS.write_text(json.dumps(diagnostics, indent=2, allow_nan=False)+"\n")
print(json.dumps(summary, indent=2))
print("Complete: saved spectra, histogram CSV, diagnostics, caption, PDF, and PNG to", OUTPUT_DIR)
print("Mixed-mode diagnostics:")
print(json.dumps(mixed_summary, indent=2))
